# Dataset mới: Barcode · Ripped · Wornout
Profile **book-three-kept-v1** bỏ Notch/Spot/Tag. Chỉ chạy sau khi duyệt và khóa split.
Notebook độc lập, không cần clone repo. Dataset từ Drive → /content; checkpoint ở Drive.
Smoke test không phải kết quả chất lượng. Test giữ kín đến khi chốt model/ngưỡng.


In [ ]:
# Chỉ cài trong runtime Colab, không đổi môi trường collector
%pip install -q ultralytics==8.4.208
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
"""Colab artifact validation and bounded experiment settings (stdlib only)."""
import json
from pathlib import Path
import hashlib, math
CLASSES=['Barcode','Ripped','Wornout']
PROFILE='book-three-kept-v1'
MAPPING={'0':0,'1':None,'2':1,'3':None,'4':None,'5':2}
def sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()
def read(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))
def labels(text, count):
    result = []
    for line in text.splitlines():
        if not line.strip():
            continue
        fields = line.split()
        if len(fields) != 5 or fields[0] not in [str(i) for i in range(count)]:
            raise ValueError("Invalid class/YOLO label")
        x, y, w, h = map(float, fields[1:])
        if not all(math.isfinite(v) and 0 <= v <= 1 for v in (x, y, w, h)) or min(w, h) <= 0:
            raise ValueError("Invalid bbox")
        result.append(" ".join(fields))
    return result

ULTRALYTICS_VERSION = '8.4.208'


def validate_split(root, expected):
    root = Path(root)
    if not expected or expected.get('taxonomy_profile') != PROFILE or not expected.get('dataset_version'):
        raise ValueError('Split not approved/locked yet')
    if sha(root/'split_manifest.json') != expected.get('split_manifest_sha256'):
        raise ValueError('Split manifest checksum mismatch')
    m = read(root/'split_manifest.json')
    if m.get('taxonomy_profile') != PROFILE or m.get('source_id_mapping') != MAPPING or m.get('classes') != CLASSES or not m.get('split_ready') or m.get('dataset_version') != expected['dataset_version']:
        raise ValueError('Wrong profile/mapping/classes/version or unlocked split')
    if not m.get('approval', {}).get('approved') or m['approval'].get('plan_sha256') != m.get('plan_sha256'):
        raise ValueError('Missing split approval')
    actual_files = {p.relative_to(root).as_posix() for p in root.rglob('*') if p.is_file()}
    if actual_files != set(m['file_sha256']) | {'split_manifest.json'}:
        raise ValueError('Split file inventory mismatch')
    for name, checksum in m['file_sha256'].items():
        p = (root/name).resolve()
        if root.resolve() not in p.parents or sha(p) != checksum:
            raise ValueError('Split file checksum/path mismatch')
    groups, ids, hashes = set(), set(), set()
    for part, rows in m['assignment'].items():
        if part not in ('train','val','test'):
            raise ValueError('Unknown split')
        local_groups = {r['group_id'] for r in rows}
        local_ids = {r['image_id'] for r in rows}
        local_hashes = {sha(root/part/'images'/r['image_file']) for r in rows}
        if groups & local_groups or ids & local_ids or hashes & local_hashes or len(local_ids) != len(rows):
            raise ValueError('Group/ID/image leakage')
        groups |= local_groups
        ids |= local_ids
        hashes |= local_hashes
        counts = {name:0 for name in CLASSES}
        for row in rows:
            for line in labels((root/part/'labels'/(row['image_id']+'.txt')).read_text(), 3):
                counts[CLASSES[int(line.split()[0])]] += 1
        if counts != m['class_counts'][part] or any(n == 0 for n in counts.values()) or len(rows) != m['counts'][part]:
            raise ValueError('Class/record counts mismatch')
    if set(m['assignment']) != {'train','val','test'} or ids & set(m['quarantined']):
        raise ValueError('Missing split or quarantine leakage')
    return m


def settings(model='n', gentle=False, seed=42, batch=16):
    if model not in ('n','s') or type(seed) is not int or seed not in (42,43,44) or batch not in (8,16):
        raise ValueError('Unknown experiment')
    config = dict(weights=f'yolo11{model}.pt', imgsz=640, epochs=100, patience=20,
                  optimizer='AdamW', lr0=.001, seed=seed, batch=batch, amp=True,
                  deterministic=True, workers=2, save=True, save_period=1,
                  name=f'{model}640_{"gentle" if gentle else "default"}_seed{seed}')
    if gentle:
        config.update(mosaic=.25, scale=.2, hsv_s=.3, hsv_v=.2)
    return config


def check_resume(run, fingerprint):
    run = Path(run)
    if read(run/'run_identity.json') != fingerprint or not (run/'train/weights/last.pt').is_file():
        raise ValueError('Resume requires same run/dataset/config and last.pt')
    return run/'train/weights/last.pt'


def compare_runs(root):
    """Three-seed comparison; model seed42 is always the deliverable."""
    import statistics
    rows = [read(p) for p in Path(root).glob('*/validation.json')]
    groups = {}
    for r in rows:
        groups.setdefault((r['dataset_version'],r['model'],r['gentle']), []).append(r)
    result = []
    for key, rs in groups.items():
        seeds = [r['seed'] for r in rs]
        if len(set(seeds)) != len(seeds):
            raise ValueError('Duplicate seed results for a configuration')
        scores = [r['macro_damage_ap5095'] for r in rs]
        result.append(dict(dataset_version=key[0], model=key[1], gentle=key[2], seeds=seeds,
                           mean=statistics.mean(scores), stdev=statistics.stdev(scores) if len(scores)>1 else None,
                           complete=set(seeds)=={42,43,44}))
    return sorted(result,key=lambda r:-r['mean'])


def operating_metrics(label_root, prediction_root, thresholds):
    """Greedy class-aware bbox matching at IoU .50, after saved NMS predictions."""
    def iou(a,b):
        def xy(r):
            x,y,w,h=r
            return x-w/2,y-h/2,x+w/2,y+h/2
        a,b=xy(a),xy(b)
        intersection=max(0,min(a[2],b[2])-max(a[0],b[0]))*max(0,min(a[3],b[3])-max(a[1],b[1]))
        union=(a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-intersection
        return intersection/union if union else 0
    totals={n:dict(tp=0,fp=0,fn=0) for n in CLASSES}
    for target in Path(label_root).glob('*.txt'):
        gt=labels(target.read_text(),3)
        pred=Path(prediction_root)/target.name
        detections=[]
        if pred.exists():
            for line in pred.read_text().splitlines():
                f=line.split()
                if len(f)!=6:
                    raise ValueError('Predictions need class xywh confidence')
                labels(' '.join(f[:5]),3)
                detections.append((int(f[0]),list(map(float,f[1:5])),float(f[5])))
        for c,name in enumerate(CLASSES):
            boxes=[list(map(float,s.split()[1:])) for s in gt if int(s.split()[0])==c]
            used=set()
            selected=sorted((d for d in detections if d[0]==c and d[2]>=thresholds[name]),key=lambda d:-d[2])
            for _,box,confidence in selected:
                candidates=[(iou(box,other),j) for j,other in enumerate(boxes) if j not in used]
                match=max(candidates,default=(0,-1))
                if match[0]>=.5:
                    used.add(match[1]); totals[name]['tp']+=1
                else:
                    totals[name]['fp']+=1
            totals[name]['fn']+=len(boxes)-len(used)
    for name,t in totals.items():
        beta=2 if name=='Ripped' else 1
        t['precision']=t['tp']/max(1,t['tp']+t['fp'])
        t['recall']=t['tp']/max(1,t['tp']+t['fn'])
        t['fbeta']=(1+beta**2)*t['tp']/max(1,(1+beta**2)*t['tp']+beta**2*t['fn']+t['fp'])
        t['threshold']=thresholds[name]
    return totals


def select_final(rows):
    """Compare finalists with complete seeds; stable tie-break favors fewer parameters."""
    import math
    complete=sorted((r for r in rows if r['complete']),key=lambda r:-r['mean'])
    if len(complete)<2 or len({r['dataset_version'] for r in complete})!=1:
        raise ValueError('Need two three-seed finalists on the same dataset')
    a,b=complete[:2]
    pooled=math.sqrt((a['stdev']**2+b['stdev']**2)/2)
    if a['mean']-b['mean']<=pooled:
        return min((a,b),key=lambda r:({'n':0,'s':1}[r['model']],r['gentle'],-r['mean']))
    return a


def choose_thresholds(label_root,prediction_root):
    grid=[.001]+[i/100 for i in range(1,100)]
    evaluated=[operating_metrics(label_root,prediction_root,{n:t for n in CLASSES}) for t in grid]
    return {n:max((row[n] for row in evaluated),key=lambda r:(r['fbeta'],-r['threshold'])) for n in CLASSES}


In [ ]:
# Dán nội dung .lock.json đã nhận SAU khi duyệt, độc lập với ZIP.
EXPECTED = None
ZIP_ON_DRIVE = '/content/drive/MyDrive/PBL5/three-kept/split.zip'
RUNS = Path('/content/drive/MyDrive/PBL5/three-kept/runs')
assert EXPECTED and EXPECTED.get('zip_sha256'), 'Chưa có split đã được duyệt/khóa; dừng trước train'
import shutil, zipfile
archive=Path('/content/three-kept.zip')
shutil.copyfile(ZIP_ON_DRIVE,archive)
assert sha(archive)==EXPECTED['zip_sha256'], 'ZIP SHA mismatch'
DATA=Path('/content/three-kept-data')
assert not DATA.exists(), 'Runtime còn dataset cũ; dùng runtime sạch hoặc kiểm tra dữ liệu'
DATA.mkdir()
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        dest=(DATA/name).resolve()
        assert DATA.resolve() in dest.parents and ':' not in name and '\\' not in name, 'Unsafe ZIP'
    z.extractall(DATA)
manifest=validate_split(DATA,EXPECTED)
runtime_yaml=Path('/content/train-three-kept.yaml')
runtime_yaml.write_text('path: '+str(DATA)+'\ntrain: train/images\nval: val/images\nnc: 3\nnames: '+json.dumps(CLASSES)+'\n')
print('PASS',manifest['dataset_version'],manifest['counts'])


## Smoke rồi chạy tối đa tám lượt
Smoke cả n/s với batch16; nếu OOM tạo run smoke mới batch8. Chọn batch chung trước các lượt đầy đủ.
Lượt1–4: n/default, s/default, n/gentle, s/gentle, seed42. Dùng validation chọn hai cấu hình;
lượt5–8 chạy hai cấu hình đó với seed43/44. Không chọn ứng viên bằng test.


In [ ]:
MODEL='n'      # n hoặc s
GENTLE=False
SEED=42
BATCH=16       # cùng batch cho toàn chiến dịch
SMOKE=True
RESUME=False
RUN_TAG='smoke-n16-01'  # tên riêng; không ghi đè run
config=settings(MODEL,GENTLE,SEED,BATCH)
if SMOKE: config.update(epochs=2,patience=2)
import platform, subprocess, torch, ultralytics
assert ultralytics.__version__==ULTRALYTICS_VERSION
assert torch.cuda.is_available(), 'Chọn GPU runtime Colab trước khi train'
env=dict(python=platform.python_version(),torch=torch.__version__,cuda=torch.version.cuda,
         gpu=torch.cuda.get_device_name(0),ultralytics=ultralytics.__version__)
identity=dict(dataset_version=manifest['dataset_version'],split_manifest_sha256=EXPECTED['split_manifest_sha256'],
              config=config,environment=env,smoke=SMOKE)
run=RUNS/RUN_TAG
from ultralytics import YOLO
if RESUME:
    checkpoint=check_resume(run,identity)
    model=YOLO(str(checkpoint))
    model.train(resume=True)
else:
    assert not run.exists(), 'Tên run đã có; resume đúng run hoặc chọn tên mới'
    previous=[read(p) for p in RUNS.glob('*/run_identity.json')] if RUNS.exists() else []
    full=[p for p in previous if not p.get('smoke')]
    if not SMOKE:
        assert len(full)<8, 'Đã đủ tám lượt chính'
        assert all(p['split_manifest_sha256']==identity['split_manifest_sha256'] and p['environment']==env and p['config']['batch']==BATCH for p in full), 'Không so trực tiếp khác dữ liệu/môi trường/batch'
    run.mkdir(parents=True)
    (run/'run_identity.json').write_text(json.dumps(identity,indent=2))
    (run/'pip-freeze.txt').write_text(subprocess.check_output(['pip','freeze'],text=True))
    shutil.copyfile(runtime_yaml,run/'runtime.yaml')
    kwargs=dict(config)
    weights=kwargs.pop('weights'); kwargs.pop('name')
    model=YOLO(weights)
    if Path(weights).is_file():
        (run/'initial-weights-sha256.txt').write_text(sha(weights))
    model.train(data=str(runtime_yaml),project=str(run),name='train',exist_ok=False,**kwargs)


In [ ]:
assert not SMOKE, 'Smoke chỉ kiểm chạy và bộ nhớ, không so chất lượng'
best=run/'train/weights/best.pt'
trained=YOLO(str(best))
metrics=trained.val(data=str(runtime_yaml),split='val',imgsz=640,batch=BATCH,project=str(run),name='validation')
rows=[]
for i,name in enumerate(CLASSES):
    p,r,ap50,ap=metrics.box.class_result(list(metrics.box.ap_class_index).index(i))
    rows.append(dict(class_name=name,precision=float(p),recall=float(r),ap50=float(ap50),ap5095=float(ap)))
result=dict(dataset_version=manifest['dataset_version'],model=MODEL,gentle=GENTLE,seed=SEED,
            macro_damage_ap5095=sum(r['ap5095'] for r in rows if r['class_name']!='Barcode')/2,
            per_class=rows,best_sha256=sha(best),environment=env)
(run/'validation.json').write_text(json.dumps(result,indent=2))
trained.predict(source=str(DATA/'val/images'),imgsz=640,conf=.001,save=True,save_txt=True,save_conf=True,
                iou=.7,project=str(run),name='val-predictions',exist_ok=True)
threshold_report=choose_thresholds(DATA/'val/labels',run/'val-predictions/labels')
(run/'thresholds-validation.json').write_text(json.dumps(threshold_report,indent=2))
print(json.dumps(compare_runs(RUNS),indent=2))


## Chốt model và test cuối
Chọn bằng macro AP50–95 hai lỗi; báo Barcode riêng. So hai cấu hình cuối qua seed42/43/44;
nếu chênh lệch không vượt độ lệch chuẩn gộp, ưu tiên model ít tham số hơn.
Bản bàn giao dùng seed42. Chọn threshold F2 Ripped, F1 Barcode/Wornout trên validation.
Không chạy test trước khi chốt model và threshold. Nếu sửa nhãn: version mới, chạy lại đối chứng.
Ô dưới chỉ mở khi đã lưu quyết định chọn model/ngưỡng; tự chạy lại test để tune là sai quy trình.


In [ ]:
# Điền sau khi hoàn tất so sánh + threshold validation
FINAL_SELECTION=None  # {'best_path':..., 'best_sha256':..., 'thresholds': {'Barcode':..., 'Ripped':..., 'Wornout':...}}
RUN_FINAL_TEST=False
if RUN_FINAL_TEST:
    assert FINAL_SELECTION and set(FINAL_SELECTION['thresholds'])==set(CLASSES)
    chosen=Path(FINAL_SELECTION['best_path'])
    assert sha(chosen)==FINAL_SELECTION['best_sha256']
    chosen_identity=read(chosen.parents[2]/'run_identity.json')
    assert chosen_identity['config']['seed']==42 and not chosen_identity['smoke']
    assert chosen_identity['split_manifest_sha256']==EXPECTED['split_manifest_sha256']
    assert all(0<=float(t)<=1 for t in FINAL_SELECTION['thresholds'].values())
    testdir=RUNS/'FINAL_TEST'
    assert not testdir.exists(), 'Test đã bắt đầu/chạy; không tune bằng test'
    testdir.mkdir()
    (testdir/'selection.json').write_text(json.dumps(FINAL_SELECTION,indent=2))
    test_yaml=Path('/content/test-three-kept.yaml')
    test_yaml.write_text(runtime_yaml.read_text()+'test: test/images\n')
    final=YOLO(str(chosen))
    final.val(data=str(test_yaml),split='test',imgsz=640,project=str(testdir),name='metrics')
    final.predict(source=str(DATA/'test/images'),imgsz=640,conf=.001,iou=.7,save_txt=True,save_conf=True,
                  project=str(testdir),name='predictions')
    report=operating_metrics(DATA/'test/labels',testdir/'predictions/labels',FINAL_SELECTION['thresholds'])
    (testdir/'operating-metrics.json').write_text(json.dumps(report,indent=2))
